# Stage 1: Importing dependencies

In [1]:
Dataset Link: http://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip

SyntaxError: invalid syntax (3333598494.py, line 1)

In [2]:
import numpy as np
import math
import re
import pandas as pd
from bs4 import BeautifulSoup


#from google.colab import drive

In [3]:

import tensorflow as tf

from tensorflow.keras import layers
import tensorflow_datasets as tfds

/opt/anaconda3/envs/DeepLearning2026/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Stage 2: Data preprocessing

## Loading files

In [4]:
import os
print(os.getcwd())
print(os.listdir())

/Users/danielfeal/Class Assignments Deep Learning
['Assignment_1_Iris_FLower_Classification.ipynb', 'Assignment_3_CiFAR_10_Images_Classification_Using_CNNs.ipynb', '.DS_Store', 'Google_Stock_Price_Train.csv', 'Assignment_7_CNN_for_NLP.ipynb', 'Assignment5_Stock Price Prediction Using RNNs.ipynb', 'cifar10-samples.png', 'Google_Stock_Price_Test.csv', '.ipynb_checkpoints', 'Assignment6_ImageCompression.ipynb', 'Assignment_4_Transfer_Learning_MNIST.ipynb', 'mnist-samples.png', 'trainingandtestdata', 'Assignment_2_MNIST Classification.ipynb']


In [5]:
#drive.mount("/content/drive")

In [6]:
cols = ["sentiment", "id", "date", "query", "user", "text"]
train_data = pd.read_csv(
    "trainingandtestdata/training.1600000.processed.noemoticon.csv",
    header=None,
    names=cols,
    engine="python",
    encoding="latin1"
)


In [7]:
train_data.head()

,sentiment,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


## Preprocessing

In [8]:
data = train_data

### Cleaning

In [9]:
data.drop(["id","date","query","user"],
          axis=1,
          inplace=True)


In [10]:
def clean_tweet(tweet):
    tweet = BeautifulSoup(tweet, "lxml").get_text()
    tweet = re.sub(r"@[A-Za-z0-9]+", ' ', tweet)
    tweet = re.sub(r"https?://[A-Za-z0-9./]+", ' ',tweet)
    tweet = re.sub(r"[^a-zA-Z.!?']", ' ', tweet)
    tweet = re.sub(r" +", ' ',tweet)
    return tweet

In [15]:
data_clean = [clean_tweet(tweet) for tweet in data.text]

/var/folders/bw/pc6h5std1ls7_6db9cjglm5r0000gn/T/ipykernel_16170/3744447478.py:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  tweet = BeautifulSoup(tweet, "lxml").get_text()


In [16]:
data_labels = data.sentiment.values
data_labels[data_labels == 4] =1

In [17]:
set(data_labels)

{0, 1}

### Tokenization

In [18]:
tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    data_clean, target_vocab_size=2**16
)
data_inputs = [tokenizer.encode(sentence) for sentence in data_clean]

### Padding

In [19]:
MAX_LEN = max([len(sentence) for sentence in data_inputs])
data_inputs = tf.keras.preprocessing.sequence.pad_sequences(data_inputs,
                                                            value=0,
                                                            padding="post",
                                                            maxlen=MAX_LEN)

### Spliting into training/testing set

In [20]:
test_idx = np.random.randint(0, 800000, 8000)
text_idx = np.concatenate((test_idx, test_idx+800000))

In [21]:
test_inputs = data_inputs[test_idx]
test_labels = data_labels[test_idx]
train_inputs = np.delete(data_inputs, test_idx, axis=0)
train_labels = np.delete(data_labels, test_idx)


# Stage 3: Model building

In [22]:
class DCNN(tf.keras.Model):
    def __init__(self,
                 vocab_size,
                 emb_dim=128,
                 nb_filters=50,
                 FFN_units=512,
                 nb_classes=2,
                 dropout_rate=0.1,
                 training=False,
                 name="dcnn"):
        super(DCNN, self).__init__(name=name)

        self.embedding = layers.Embedding(vocab_size,
        emb_dim)
        
        self.bigram = layers.Conv1D(filters=nb_filters,
        kernel_size=2,
        padding="valid",
        activation="relu")
        self.pool_1 = layers.GlobalMaxPool1D()
        self.trigram = layers.Conv1D(filters=nb_filters,
        kernel_size=3,
        padding="valid",
        activation="relu")
        self.pool_2 = layers.GlobalMaxPool1D()
        self.fourgram = layers.Conv1D(filters=nb_filters,
        kernel_size=4,
        padding="valid",
        activation="relu")
        self.pool_3 = layers.GlobalMaxPool1D()
        self.dense_1 = layers.Dense(units=FFN_units, activation="relu")
        self.dropout = layers.Dropout(rate=dropout_rate)
        if nb_classes ==2:
            self.last_dense = layers.Dense(units=1,
                                           activation="sigmoid")
        else:
            self.last_dense = layers.Dense(units=nb_classes,
                                           activation= "softmax")

    def call(self, inputs, training):
        x = self.embedding(inputs)
        x_1 = self.bigram(x)
        x_1 = self.pool_1(x_1)
        x_2 = self.trigram(x)
        x_2 = self.pool_2(x_2)
        x_3 = self.fourgram(x)
        x_3 = self.pool_3(x_3)

        merged = tf.concat([x_1, x_2, x_3], axis=-1)
        merged = self.dense_1(merged)
        merged = self.dropout(merged, training)
        output = self.last_dense(merged)

        return output

# Stage 4: Application

## Config

In [23]:
VOCAB_SIZE = tokenizer.vocab_size

EMB_DIM = 200
NB_FILTERS = 100
FFN_UNITS = 256
NB_CLASSES = len(set(train_labels))

DROPOUT_RATE = 0.2

BATCH_SIZE = 32
NB_EPOCHS = 5

## Training

In [24]:
Dcnn = DCNN(vocab_size=VOCAB_SIZE,
            emb_dim=EMB_DIM,
            nb_filters=NB_FILTERS,
            FFN_units=FFN_UNITS,
            nb_classes=NB_CLASSES,
            dropout_rate=DROPOUT_RATE)

In [25]:
if NB_CLASSES == 2:
    Dcnn.compile(loss="binary_crossentropy",
                 optimizer="adam",
                 metrics=["accuracy"])
else:
    Dcnn.compile(loss="sparse_categorical_crossentropy",
                 optimizer="adam",
                 metrics=["sparse_categorical_accuracy"])

In [26]:
checkpoint_path = "./ckpt"

ckpt = tf.train.Checkpoint(Dcnn=Dcnn)

ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=1)

if ckpt_manager.latest_checkpoint:
    ckpt.restore(ckpt_manager.latest_checkpoint)
    print("Latest checkpoint restored!")

In [27]:
Dcnn.fit(train_inputs,
         train_labels,
         batch_size=BATCH_SIZE,
         epochs=NB_EPOCHS)
ckpt_manager.save()

Epoch 1/5


2026-02-01 19:17:33.054678: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


49752/49752 [==============================] - 2361s 47ms/step - loss: 0.3987 - accuracy: 0.8200
Epoch 2/5
49752/49752 [==============================] - 2365s 48ms/step - loss: 0.3339 - accuracy: 0.8568
Epoch 3/5
49752/49752 [==============================] - 2273s 46ms/step - loss: 0.2824 - accuracy: 0.8832
Epoch 4/5
49752/49752 [==============================] - 2339s 47ms/step - loss: 0.2301 - accuracy: 0.9072
Epoch 5/5
49752/49752 [==============================] - 2246s 45ms/step - loss: 0.1859 - accuracy: 0.9267


'./ckpt/ckpt-1'

## Evaluation

In [28]:
Dcnn(np.array([tokenizer.encode("You are so nice")]), training=False).numpy()

array([[0.9909577]], dtype=float32)